## Matrix solve

In fact, `expected_sojourn_time` produce the same result as R's solve. If we start from the matrices:

In [ ]:
import phasic
from phasic import Graph, StateIndexer, Property, configure
from scipy.linalg import solve
import time
phasic.configure(graph_cache=True)

def two_locus_arg_2param(state, indexer=None): 

    transitions = []
    if state.sum() <= 1: return transitions

    for i in range(indexer.state_length):
        if state[i] == 0: continue
        pi = indexer.index_to_props(i)

        for j in range(i, indexer.state_length):
            if state[j] == 0: continue
            pj = indexer.index_to_props(j)
            
            same = int(i == j)
            if same and state[i] < 2:
                continue
            if not same and (state[i] < 1 or state[j] < 1):
                continue 
            child = state.copy()
            child[i] -= 1
            child[j] -= 1
            loc1 = pi.descendants.loc1 + pj.descendants.loc1
            loc2 = pi.descendants.loc2 + pj.descendants.loc2
            if loc1 <= nr_samples and loc2 <= nr_samples:
                child[indexer.props_to_index(loc1=loc1, loc2=loc2)] += 1
                transitions.append([child, [state[i]*(state[j]-same)/(1+same), 0]])

        if state[i] > 0 and pi.descendants.loc1 > 0 and pi.descendants.loc2 > 0:
            child = state.copy()
            child[i] -= 1
            child[indexer.props_to_index(loc1=pi.descendants.loc1, loc2=0)] += 1
            child[indexer.props_to_index(loc1=0, loc2=pi.descendants.loc2)] += 1
            transitions.append([child, [0, 1]])

    return transitions

import timeit
import sys

try:
    min_n = int(sys.argv[1])
    max_n = int(sys.argv[2])
except:
    min_n = 3
    max_n = 4

print(f'n\tsize\tsolve\texp.\tp.exp.\tsoj.\tp.soj.')

for nr_samples in range(min_n, max_n+1):
    indexer = StateIndexer(descendants=[
        Property('loc1', max_value=nr_samples),
        Property('loc2', max_value=nr_samples)
    ])
    initial = [0] * indexer.state_length
    initial[indexer.props_to_index(loc1=1, loc2=1)] = nr_samples
    graph = Graph(two_locus_arg_2param, ipv=initial, indexer=indexer) 
    mats = graph.as_matrices()
    ss = timeit.timeit(lambda : -solve(mats.sim.T, mats.ipv), number=100)
    with configure(parallel_elimination=False):
        e = timeit.timeit(lambda : graph.expected_sojourn_time(), number=100)
    with configure(parallel_elimination=True):
        pe = timeit.timeit(lambda : graph.expected_sojourn_time(), number=100)
    with configure(parallel_elimination=False):
        est = timeit.timeit(lambda : graph.expected_sojourn_time(), number=100)
    with configure(parallel_elimination=True):
        pest = timeit.timeit(lambda : graph.expected_sojourn_time(), number=100)
    print(f'{nr_samples}\t{graph.vertices_length()}\t{ss:.3f}\t{e:.3f}\t{pe:.3f}\t{est:.3f}\t{pest:.3f}')

n	size	solve	exp.	p.exp.	soj.	p.soj.
3	32	0.004	0.002	0.001	0.001	0.001
4	110	0.027	0.021	0.014	0.012	0.012
5	340	0.251	0.176	0.170	0.169	0.167
6	1044	1.519	3.019	1.074	2.769	1.000
7	2999	15.812	41.190	11.177	40.313	11.400


`expected_sojourn_time` on a graph made from the matrices produces this:

In [ ]:
%%monitor
graph.update_weights([1.0, 1.0])
x = graph.expected_sojourn_time()

In [ ]:
%%time
x = graph.expected_sojourn_time()

CPU times: user 510 ms, sys: 27.8 ms, total: 538 ms
Wall time: 538 ms


In [ ]:
%%time
x = graph.expected_sojourn_time()

CPU times: user 511 ms, sys: 25.2 ms, total: 536 ms
Wall time: 536 ms


In [ ]:
from scipy.linalg import solve
mat = graph.as_matrices()
x = -solve(mat.sim.T, mat.ipv)
x

In [ ]:
%%time
x = -solve(SIM.T, IPV)

CPU times: user 537 ms, sys: 47.5 ms, total: 585 ms
Wall time: 183 ms
